# Finance Q&A Finetune — Colab Notebook

Finetunes **Qwen2.5-3B-Instruct** on the **gbharti/finance-alpaca** dataset using Unsloth (QLoRA, r=16).

**Before you start:**
1. `Runtime → Change runtime type → T4 GPU` (free tier is fine).
2. Have your HuggingFace access token ready: https://huggingface.co/settings/tokens (a *read* token is enough).
3. Run cells in order, top to bottom.

**Expected runtime on free T4:** ~45 minutes end-to-end for the 10,000-row default. Set `SUBSET_SIZE = None` in section 3 for the full 68k dataset (requires Colab Pro / A100 or ~4 hours patience).

## 0. Pre-flight — confirm the GPU

You should see a `Tesla T4` with ~15 GB. If you get an error or no GPU, fix the runtime type before going further.

In [ ]:
!nvidia-smi

## 1. Install Unsloth

Takes ~2 minutes. The `%%capture` hides the pip spam; if it fails, remove that line and rerun to see errors.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps "git+https://github.com/unslothai/unsloth.git"

## 2. Log into HuggingFace

Paste your token in the prompt. Needed to download the base model and the dataset.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Configure the run

- `MODEL_NAME` is set to Qwen2.5-3B — safe for T4's 15 GB VRAM.
- `SUBSET_SIZE = 10000` keeps training to ~30–45 min on a free T4. Set to `None` for the full 68k dataset.

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048
DATASET_NAME = "gbharti/finance-alpaca"
OUTPUT_DIR = "lora_finance"
SUBSET_SIZE = 10000  # None = full dataset

## 4. Load the base model (4-bit)

Downloads the quantized Qwen2.5-3B (~2 GB). First run takes a minute; cached after.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

## 5. Attach LoRA adapters

LoRA adds small trainable matrices (~0.1% of the model's params) on top of the frozen base weights. Training only updates those small matrices — that's why it's so much cheaper than full finetuning.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 6. Load and format the dataset

Each row has `instruction`, `input` (often empty), and `output`. We render it into a Qwen chat-template conversation. ~28% of rows have a non-empty `input` (extra context); we fold that into the user turn.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, split="train")
if SUBSET_SIZE:
    dataset = dataset.shuffle(seed=3407).select(range(SUBSET_SIZE))
print(f"Training on {len(dataset):,} rows")

def row_to_messages(row):
    instruction = row["instruction"].strip()
    extra = (row.get("input") or "").strip()
    user = f"{instruction}\n\n{extra}" if extra else instruction
    return [
        {"role": "user", "content": user},
        {"role": "assistant", "content": row["output"].strip()},
    ]

def format_row(row):
    text = tokenizer.apply_chat_template(
        row_to_messages(row), tokenize=False, add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_row, remove_columns=dataset.column_names)
print("\nExample formatted row:\n")
print(dataset[0]["text"][:500])

## 7. Train

Watch the `loss` column in the output — it should drop from ~1.8–2.2 down toward ~1.0–1.3 over the run. If it stays flat, something's off (usually a chat-template mismatch).

**Expected time on free T4:** ~30–45 min for 10k rows.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=SFTConfig(
        output_dir="outputs",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        logging_steps=10,
        save_strategy="no",
        bf16=is_bfloat16_supported(),
        fp16=not is_bfloat16_supported(),
        report_to="none",
        seed=3407,
    ),
)
trainer.train()

## 8. Save the adapter locally

Saves the LoRA weights to `./lora_finance/`. **Colab wipes all files when the session ends** — use section 8b to persist to Drive.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
!ls -lh lora_finance

## 8b. (Recommended) Copy the adapter to Google Drive

Mounts your Google Drive and copies `lora_finance/` into it, so it survives after the Colab session ends. Skip this cell if you prefer to just download the adapter in the next section.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r lora_finance /content/drive/MyDrive/
print("Saved to Drive: /content/drive/MyDrive/lora_finance")

## 9. Evaluate the finetuned model

Runs 7 fixed finance prompts through the finetuned model. Section 10 then runs the **same prompts** on the un-finetuned base model so you can compare. That comparison is your main deliverable.

In [ ]:
FastLanguageModel.for_inference(model)

PROMPTS = [
    "Explain what a P/E ratio is and why it matters to an investor.",
    "What is the difference between a Roth IRA and a Traditional IRA?",
    "How does dollar-cost averaging work, and when is it a bad idea?",
    "What is an ETF, and how is it different from a mutual fund?",
    "Is it better to pay off a low-interest mortgage early or invest the extra money?",
    "Explain bond duration in simple terms.",
    "What are the tax implications of selling a stock I've held for 6 months?",
]

def answer(model, tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to(model.device)
    out = model.generate(
        input_ids=inputs, max_new_tokens=400,
        do_sample=False, pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

finetuned_answers = []
for prompt in PROMPTS:
    response = answer(model, tokenizer, prompt)
    finetuned_answers.append((prompt, response))
    print(f"\nQ: {prompt}\nA: {response}\n{'-'*60}")

## 10. Evaluate the base model (same prompts)

Frees the finetuned model and reloads the base model fresh — T4 doesn't have room for both at once. Runs the same 7 prompts so we can diff the answers.

In [ ]:
import gc, torch
del model, tokenizer, trainer
gc.collect(); torch.cuda.empty_cache()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True,
)
base_tokenizer = get_chat_template(base_tokenizer, chat_template="qwen-2.5")
FastLanguageModel.for_inference(base_model)

base_answers = []
for prompt in PROMPTS:
    response = answer(base_model, base_tokenizer, prompt)
    base_answers.append((prompt, response))
    print(f"\nQ: {prompt}\nA: {response}\n{'-'*60}")

## 11. Write the side-by-side comparison

Produces `results_comparison.md` and downloads it. This is the primary artifact for your project write-up.

In [ ]:
with open("results_comparison.md", "w") as f:
    f.write("# Before vs After — Finance Q&A Finetune\n\n")
    f.write(f"Base model: `{MODEL_NAME}` | Dataset: `{DATASET_NAME}` | Training rows: {SUBSET_SIZE or 'full'}\n\n")
    for (prompt, ft), (_, base) in zip(finetuned_answers, base_answers):
        f.write(f"## {prompt}\n\n")
        f.write(f"**Base model:**\n\n{base}\n\n")
        f.write(f"**Finetuned:**\n\n{ft}\n\n---\n\n")

from google.colab import files
files.download("results_comparison.md")

## Done.

- Your LoRA adapter is at `./lora_finance/` (and `/content/drive/MyDrive/lora_finance/` if you ran 8b).
- The comparison is downloaded as `results_comparison.md`.

**What to include in your project report:**
1. GPU + environment info (first cell's output).
2. A few loss values from training to show it decreased.
3. 2–3 prompts from `results_comparison.md` where the finetune clearly wins, and 1 where it didn't — with your analysis.
4. What you'd do differently (more data? bigger model? more epochs?).

**Troubleshooting:**
- *OOM in section 7:* set `per_device_train_batch_size=1`.
- *Loss stays flat:* chat template mismatch — verify `get_chat_template(..., "qwen-2.5")` is called in section 4.
- *NaN loss on T4:* shouldn't happen since `is_bfloat16_supported()` auto-falls-back to fp16, but if it does, re-run the training cell (first-step NaNs sometimes self-resolve).